In [ ]:
# train_model.py

import torch
from ultralytics import YOLO

# 1. GPU(CUDA) 사용 가능 여부 확인
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"현재 사용 장치: {device}")

# 2. YOLOv10 모델 불러오기
# 처음 실행 시 yolov10m.pt가 없으면 다운로드가 필요할 수 있음
model = YOLO('yolov10m.pt')

# 3. 모델 학습 시작
print("모델 학습을 시작합니다...")

results = model.train(
    data='dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='Adam',
    save_period=10,
    name='train_run1',
    device=device,
    workers=4
)

print("\n모델 학습이 완료되었습니다.")

In [ ]:
# test_webcam.py

import cv2
import time
import torch
from pathlib import Path
from ultralytics import YOLO

# ==============================
# 1. 설정
# ==============================

# 본인 best.pt 경로로 수정하세요.
MODEL_PATH = r"D:\runs\detect\train_run14\weights\best.pt"

# 노트북 기본 웹캠이면 보통 0
# 외장 웹캠이면 1 또는 2일 수 있음
CAMERA_INDEX = 0

# confidence threshold
CONF_THRES = 0.25

# 입력 이미지 크기
IMG_SIZE = 640


# ==============================
# 2. GPU 확인
# ==============================

device = 0 if torch.cuda.is_available() else "cpu"

print("현재 사용 장치:", "cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))


# ==============================
# 3. 모델 로드
# ==============================

model_path = Path(MODEL_PATH)

if not model_path.exists():
    raise FileNotFoundError(f"best.pt 파일을 찾을 수 없습니다: {model_path}")

model = YOLO(str(model_path))

print("모델 로드 완료:", model_path)


# ==============================
# 4. 웹캠 열기
# ==============================

# Windows에서는 cv2.CAP_DSHOW를 붙이면 웹캠 열림이 더 안정적인 경우가 많습니다.
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)

if not cap.isOpened():
    raise RuntimeError(
        f"웹캠을 열 수 없습니다. CAMERA_INDEX={CAMERA_INDEX} 값을 0, 1, 2로 바꿔보세요."
    )

# 웹캠 해상도 설정
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("웹캠 실행 중입니다. 종료하려면 q 키를 누르세요.")


# ==============================
# 5. 실시간 추론
# ==============================

prev_time = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        print("프레임을 읽지 못했습니다.")
        break

    # YOLO 추론
    results = model.predict(
        source=frame,
        imgsz=IMG_SIZE,
        conf=CONF_THRES,
        device=device,
        verbose=False
    )

    # 탐지 결과가 그려진 이미지
    annotated_frame = results[0].plot()

    # FPS 계산
    current_time = time.time()
    fps = 1.0 / (current_time - prev_time)
    prev_time = current_time

    # FPS 화면 표시
    cv2.putText(
        annotated_frame,
        f"FPS: {fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # 화면 출력
    cv2.imshow("YOLO best.pt Webcam Test", annotated_frame)

    # q 또는 ESC 누르면 종료
    key = cv2.waitKey(1) & 0xFF

    if key == ord("q") or key == 27:
        break


# ==============================
# 6. 종료 처리
# ==============================

cap.release()
cv2.destroyAllWindows()

print("웹캠 테스트 종료")